In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Clone official Microsoft BEATs implementation
from pathlib import Path
import sys, os

BEATS_REPO_DIR = Path('/content/unilm')
BEATS_CODE_DIR = BEATS_REPO_DIR / 'beats'

if not BEATS_CODE_DIR.exists():
    !git clone --depth 1 https://github.com/microsoft/unilm.git /content/unilm

sys.path.append(str(BEATS_CODE_DIR))
print('BEATs code directory:', BEATS_CODE_DIR)
print('Exists:', BEATS_CODE_DIR.exists())

Cloning into '/content/unilm'...
remote: Enumerating objects: 6250, done.
remote: Counting objects: 100% (6250/6250), done.
remote: Compressing objects: 100% (3780/3780), done.
remote: Total 6250 (delta 2344), reused 5942 (delta 2233), pack-reused 0 (from 0)
Receiving objects: 100% (6250/6250), 60.34 MiB | 607.00 KiB/s, done.
Resolving deltas: 100% (2344/2344), done.
BEATs code directory: /content/unilm/beats
Exists: True


In [5]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torchaudio
from torch.utils.data import Dataset, DataLoader

from BEATs import BEATs, BEATsConfig

import zipfile

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Dataset paths and checkpoint

Update these paths to match your shared Drive folder.

Expected structure:

```text
ESC Project Deep Learning 5888 Data/
├── audio/
├── meta/esc50.csv
├── checkpoints/BEATs_iter3_plus_AS2M.pt
└── results/
```

Download the BEATs checkpoint manually from the official Microsoft BEATs README and place it in the `checkpoints` folder. Recommended: **BEATs_iter3+ (AS2M) pre-trained model**.


In [6]:
ZIP_PATH = '/content/drive/MyDrive/ESCProjectDeepLearning5888DataDump/piczak_dataset.zip'
EXTRACT_DIR = '/content/piczak_dataset'
OUTPUT_DIR = '/content/drive/MyDrive/5888_Project/results/Phase1_Baseline'

os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.isdir(EXTRACT_DIR):
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_file:
        zip_file.extractall(EXTRACT_DIR)

CSV_PATH = None
AUDIO_DIR = None

for root, dirs, files in os.walk(EXTRACT_DIR):
    if 'esc50.csv' in files:
        CSV_PATH = os.path.join(root, 'esc50.csv')

    wav_files = [file_name for file_name in files if file_name.endswith('.wav')]
    if len(wav_files) > 0 and AUDIO_DIR is None:
        AUDIO_DIR = root

print('Dataset CSV:  ', CSV_PATH)

df = pd.read_csv(CSV_PATH)

display(df.head())

Dataset CSV:   /content/piczak_dataset/esc50.csv


,filename,fold,target,category,esc10,src_file,take
0,1-100032-A-0.wav,1,0,dog,True,100032,A
1,1-100038-A-14.wav,1,14,chirping_birds,False,100038,A
2,1-100210-A-36.wav,1,36,vacuum_cleaner,False,100210,A
3,1-100210-B-36.wav,1,36,vacuum_cleaner,False,100210,B
4,1-101296-A-19.wav,1,19,thunderstorm,False,101296,A


In [16]:
BEATS_CONFIG = {
    'model_name': 'BEATs',
    'variant_name': 'real_beats_finetune_fold1',
    'checkpoint_path': '/content/drive/MyDrive/ESCProjectDeepLearning5888DataDump/checkpoints/BEATs_iter3_plus_AS2M.pt',

    'sample_rate': 16000,
    'clip_seconds': 5.0,
    'num_classes': 50,

    'num_epochs': 10,
    'batch_size': 4,
    'learning_rate': 1e-5,
    'weight_decay': 1e-4,
    'grad_clip_norm': 1.0,

    # True = only train classifier head; False = full fine-tuning
    'freeze_backbone': False,

    'save_dir': '/content/drive/MyDrive/ESCProjectDeepLearning5888DataDump/BEATs/results'
}

## ESC-50 waveform dataset for BEATs

BEATs uses 16 kHz waveform input. This notebook trains on full 5-second clips, so the reported `test_acc` is already **clip-level accuracy**.


In [9]:
class ESC50BEATsDataset(Dataset):
    def __init__(self, dataframe, audio_dir, sample_rate=16000, clip_seconds=5.0):
        self.dataframe = dataframe.reset_index(drop=True)
        self.audio_dir = Path(audio_dir)
        self.sample_rate = sample_rate
        self.target_num_samples = int(sample_rate * clip_seconds)

    def __len__(self):
        return len(self.dataframe)

    def _load_audio(self, filename):
        audio_path = self.audio_dir / filename
        waveform, sr = torchaudio.load(audio_path)

        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        if sr != self.sample_rate:
            waveform = torchaudio.transforms.Resample(sr, self.sample_rate)(waveform)

        waveform = waveform.squeeze(0)

        if waveform.numel() < self.target_num_samples:
            waveform = torch.nn.functional.pad(waveform, (0, self.target_num_samples - waveform.numel()))
        elif waveform.numel() > self.target_num_samples:
            waveform = waveform[:self.target_num_samples]

        return waveform.float()

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        waveform = self._load_audio(row['filename'])
        label = int(row['target'])
        filename = row['filename']
        return waveform, label, filename


def beats_collate_fn(batch):
    waveforms, labels, filenames = zip(*batch)
    waveforms = torch.stack(waveforms, dim=0)  # [B, T]
    labels = torch.tensor(labels, dtype=torch.long)
    padding_mask = torch.zeros(waveforms.shape, dtype=torch.bool)
    return waveforms, padding_mask, labels, filenames


def get_beats_train_test_loaders(dataframe, audio_dir, test_fold, config, num_workers=2):
    train_df = dataframe[dataframe['fold'] != test_fold].reset_index(drop=True)
    test_df = dataframe[dataframe['fold'] == test_fold].reset_index(drop=True)

    train_dataset = ESC50BEATsDataset(train_df, audio_dir, config['sample_rate'], config['clip_seconds'])
    test_dataset = ESC50BEATsDataset(test_df, audio_dir, config['sample_rate'], config['clip_seconds'])

    train_loader = DataLoader(
        train_dataset,
        batch_size=config['batch_size'],
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
        collate_fn=beats_collate_fn
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=config['batch_size'],
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        collate_fn=beats_collate_fn
    )

    return train_loader, test_loader

## Real BEATs model wrapper

This loads the official BEATs checkpoint, extracts frame-level transformer features, mean-pools them over time, and trains a new 50-class ESC-50 head.


In [10]:
class BEATsESC50Classifier(nn.Module):
    def __init__(self, checkpoint_path, num_classes=50, freeze_backbone=False):
        super().__init__()

        checkpoint_path = Path(checkpoint_path)
        if not checkpoint_path.exists():
            raise FileNotFoundError(
                f'BEATs checkpoint not found: {checkpoint_path}\n'
                'Download a BEATs pretrained checkpoint and update BEATS_CKPT_PATH.'
            )

        checkpoint = torch.load(checkpoint_path, map_location='cpu')
        cfg = BEATsConfig(checkpoint['cfg'])

        self.beats = BEATs(cfg)
        missing, unexpected = self.beats.load_state_dict(checkpoint['model'], strict=False)
        print('Loaded BEATs checkpoint')
        print('Missing keys:', len(missing))
        print('Unexpected keys:', len(unexpected))

        self.hidden_dim = cfg.encoder_embed_dim

        if freeze_backbone:
            for p in self.beats.parameters():
                p.requires_grad = False

        self.classifier = nn.Sequential(
            nn.LayerNorm(self.hidden_dim),
            nn.Dropout(0.2),
            nn.Linear(self.hidden_dim, num_classes)
        )

    def forward(self, waveforms, padding_mask=None):
        features = self.beats.extract_features(waveforms, padding_mask=padding_mask)[0]

        if features.dim() == 3:
            pooled = features.mean(dim=1)
        elif features.dim() == 2 and features.shape[1] == self.hidden_dim:
            pooled = features
        else:
            raise ValueError(
                f'Unexpected BEATs feature shape: {features.shape}. '
                'Use a pretrained BEATs checkpoint, not an AudioSet classification-head checkpoint.'
            )

        return self.classifier(pooled)


def build_beats_from_config(config, device=DEVICE):
    model = BEATsESC50Classifier(
        checkpoint_path=config['checkpoint_path'],
        num_classes=config['num_classes'],
        freeze_backbone=config['freeze_backbone']
    )
    return model.to(device)

In [11]:
def evaluate_beats(model, data_loader, criterion, device=DEVICE):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for waveforms, padding_mask, labels, filenames in data_loader:
            waveforms = waveforms.to(device)
            padding_mask = padding_mask.to(device)
            labels = labels.to(device)

            logits = model(waveforms, padding_mask=padding_mask)
            loss = criterion(logits, labels)

            total_loss += loss.item() * labels.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return {
        'loss': total_loss / total,
        'acc': correct / total
    }


def run_model_beats_real(model, train_loader, test_loader, config, device=DEVICE):
    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=config['learning_rate'],
        weight_decay=config['weight_decay']
    )

    history = {
        'train_loss': [],
        'train_acc': [],
        'test_loss': [],
        'test_acc': []
    }

    best_test_acc = -1.0
    best_epoch = -1
    best_state = None

    start_time = time.time()

    for epoch in range(config['num_epochs']):
        model.train()
        train_loss_total = 0.0
        train_correct = 0
        train_total = 0

        for waveforms, padding_mask, labels, filenames in train_loader:
            waveforms = waveforms.to(device)
            padding_mask = padding_mask.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            logits = model(waveforms, padding_mask=padding_mask)
            loss = criterion(logits, labels)
            loss.backward()

            if config.get('grad_clip_norm') is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), config['grad_clip_norm'])

            optimizer.step()

            train_loss_total += loss.item() * labels.size(0)
            preds = logits.argmax(dim=1)
            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)

        train_loss = train_loss_total / train_total
        train_acc = train_correct / train_total

        test_metrics = evaluate_beats(model, test_loader, criterion, device=device)
        test_loss = test_metrics['loss']
        test_acc = test_metrics['acc']

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_loss'].append(test_loss)
        history['test_acc'].append(test_acc)

        if test_acc > best_test_acc:
            best_test_acc = test_acc
            best_epoch = epoch + 1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        print(
            f"[BEATs] Epoch {epoch+1}/{config['num_epochs']} | "
            f"Train Loss={train_loss:.4f}, Train Acc={train_acc:.4f} | "
            f"Test Loss={test_loss:.4f}, Test Acc={test_acc:.4f}"
        )

    training_time_sec = time.time() - start_time

    if best_state is not None:
        model.load_state_dict(best_state)

    results = {
        'model': config['model_name'],
        'variant': config['variant_name'],
        'checkpoint_path': config['checkpoint_path'],
        'freeze_backbone': config['freeze_backbone'],
        'final_train_loss': history['train_loss'][-1],
        'final_train_acc': history['train_acc'][-1],
        'final_test_loss': history['test_loss'][-1],
        'final_test_acc': history['test_acc'][-1],
        'best_test_acc': best_test_acc,
        'best_epoch': best_epoch,
        'training_time_sec': training_time_sec,
        'num_trainable_params': sum(p.numel() for p in model.parameters() if p.requires_grad),
        'num_total_params': sum(p.numel() for p in model.parameters())
    }

    return history, results

In [12]:
def save_beats_results(history, results, config, model=None):
    save_path = Path(config['save_dir']) / config['model_name'] / config['variant_name']
    save_path.mkdir(parents=True, exist_ok=True)

    pd.DataFrame(history).to_csv(save_path / 'history.csv', index=False)
    pd.DataFrame([results]).to_csv(save_path / 'results.csv', index=False)

    with open(save_path / 'config.json', 'w') as f:
        json.dump(config, f, indent=2)

    plt.figure(figsize=(8, 5))
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['test_loss'], label='Test Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(f"{config['model_name']} - {config['variant_name']} Loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path / 'loss_curves.png', dpi=200, bbox_inches='tight')
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.plot(history['train_acc'], label='Train Acc')
    plt.plot(history['test_acc'], label='Test Acc')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.title(f"{config['model_name']} - {config['variant_name']} Accuracy")
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path / 'accuracy_curves.png', dpi=200, bbox_inches='tight')
    plt.close()

    if model is not None:
        torch.save(model.state_dict(), save_path / 'best_model.pt')

    print('Saved BEATs results to:', save_path)

In [13]:
def run_beats_experiment(config, dataframe, audio_dir, test_fold, device=DEVICE):
    print(f"Running {config['model_name']} | {config['variant_name']} | fold {test_fold}")

    train_loader, test_loader = get_beats_train_test_loaders(
        dataframe=dataframe,
        audio_dir=audio_dir,
        test_fold=test_fold,
        config=config
    )

    model = build_beats_from_config(config, device=device)

    history, results = run_model_beats_real(
        model=model,
        train_loader=train_loader,
        test_loader=test_loader,
        config=config,
        device=device
    )

    save_beats_results(history, results, config, model=model)

    return history, results

## One-fold evaluation


In [17]:
history_beats, results_beats = run_beats_experiment(
    config=BEATS_CONFIG,
    dataframe=df,
    audio_dir=AUDIO_DIR,
    test_fold=1,
    device=DEVICE
)

results_beats

Running BEATs | real_beats_finetune_fold1 | fold 1


/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Loaded BEATs checkpoint
Missing keys: 0
Unexpected keys: 0
[BEATs] Epoch 1/10 | Train Loss=3.1088, Train Acc=0.3100 | Test Loss=1.9312, Test Acc=0.6900
[BEATs] Epoch 2/10 | Train Loss=1.3027, Train Acc=0.8269 | Test Loss=0.6801, Test Acc=0.9125
[BEATs] Epoch 3/10 | Train Loss=0.5086, Train Acc=0.9463 | Test Loss=0.3483, Test Acc=0.9200
[BEATs] Epoch 4/10 | Train Loss=0.2398, Train Acc=0.9781 | Test Loss=0.2905, Test Acc=0.9250
[BEATs] Epoch 5/10 | Train Loss=0.1171, Train Acc=0.9900 | Test Loss=0.2810, Test Acc=0.9300
[BEATs] Epoch 6/10 | Train Loss=0.0762, Train Acc=0.9925 | Test Loss=0.2669, Test Acc=0.9325
[BEATs] Epoch 7/10 | Train Loss=0.0526, Train Acc=0.9962 | Test Loss=0.2685, Test Acc=0.9275
[BEATs] Epoch 8/10 | Train Loss=0.0494, Train Acc=0.9969 | Test Loss=0.2691, Test Acc=0.9400
[BEATs] Epoch 9/10 | Train Loss=0.0422, Train Acc=0.9981 | Test Loss=0.2583, Test Acc=0.9300
[BEATs] Epoch 10/10 | Train Loss=0.0235, Train Acc=0.9975 | Test Loss=0.2515, Test Acc=0.9475
Saved BEAT

{'model': 'BEATs',
 'variant': 'real_beats_finetune_fold1',
 'checkpoint_path': '/content/drive/MyDrive/ESCProjectDeepLearning5888DataDump/checkpoints/BEATs_iter3_plus_AS2M.pt',
 'freeze_backbone': False,
 'final_train_loss': 0.023459364423251826,
 'final_train_acc': 0.9975,
 'final_test_loss': 0.25153378237911966,
 'final_test_acc': 0.9475,
 'best_test_acc': 0.9475,
 'best_epoch': 10,
 'training_time_sec': 152.65110969543457,
 'num_trainable_params': 90351778,
 'num_total_params': 90351778}

## Five-fold evaluation

Each official ESC-50 fold is used once as the test set. This produces paper-comparable clip-level accuracy across all folds.


In [18]:
def run_beats_5_fold_evaluation(config, dataframe, audio_dir, device=DEVICE):
    all_results = []
    all_histories = {}

    for fold in sorted(dataframe['fold'].unique()):
        fold_config = config.copy()
        fold_config['variant_name'] = f"real_beats_finetune_fold{fold}"

        print('=' * 80)
        print(f"Running BEATs 5-fold evaluation: fold {fold}")
        print('=' * 80)

        history, results = run_beats_experiment(
            config=fold_config,
            dataframe=dataframe,
            audio_dir=audio_dir,
            test_fold=fold,
            device=device
        )

        results['fold'] = fold
        all_results.append(results)
        all_histories[f'fold_{fold}'] = history

    results_df = pd.DataFrame(all_results)

    summary = {
        'model': config['model_name'],
        'checkpoint_path': config['checkpoint_path'],
        'num_folds': len(results_df),
        'mean_test_acc': results_df['final_test_acc'].mean(),
        'std_test_acc': results_df['final_test_acc'].std(),
        'mean_best_test_acc': results_df['best_test_acc'].mean(),
        'std_best_test_acc': results_df['best_test_acc'].std(),
        'mean_test_loss': results_df['final_test_loss'].mean(),
        'std_test_loss': results_df['final_test_loss'].std(),
        'mean_training_time_sec': results_df['training_time_sec'].mean()
    }

    save_path = Path(config['save_dir']) / config['model_name'] / 'real_beats_5fold_summary'
    save_path.mkdir(parents=True, exist_ok=True)

    results_df.to_csv(save_path / 'fold_results.csv', index=False)
    pd.DataFrame([summary]).to_csv(save_path / 'summary.csv', index=False)

    with open(save_path / 'summary.json', 'w') as f:
        json.dump(summary, f, indent=2)

    print('\n5-Fold BEATs Summary')
    print('--------------------')
    print(f"Mean Final Test Accuracy: {summary['mean_test_acc']:.4f}")
    print(f"Std Final Test Accuracy:  {summary['std_test_acc']:.4f}")
    print(f"Mean Best Test Accuracy:  {summary['mean_best_test_acc']:.4f}")
    print(f"Std Best Test Accuracy:   {summary['std_best_test_acc']:.4f}")
    print(f"Saved summary to: {save_path}")

    return results_df, summary, all_histories

In [19]:
beats_5fold_results, beats_5fold_summary, beats_5fold_histories = run_beats_5_fold_evaluation(
    config=BEATS_CONFIG,
    dataframe=df,
    audio_dir=AUDIO_DIR,
    device=DEVICE
)

beats_5fold_summary

Running BEATs 5-fold evaluation: fold 1
Running BEATs | real_beats_finetune_fold1 | fold 1
Loaded BEATs checkpoint
Missing keys: 0
Unexpected keys: 0
[BEATs] Epoch 1/10 | Train Loss=3.1108, Train Acc=0.3525 | Test Loss=1.8812, Test Acc=0.7775
[BEATs] Epoch 2/10 | Train Loss=1.1791, Train Acc=0.8688 | Test Loss=0.6502, Test Acc=0.8825
[BEATs] Epoch 3/10 | Train Loss=0.4141, Train Acc=0.9600 | Test Loss=0.3310, Test Acc=0.9325
[BEATs] Epoch 4/10 | Train Loss=0.2151, Train Acc=0.9781 | Test Loss=0.2590, Test Acc=0.9350
[BEATs] Epoch 5/10 | Train Loss=0.1141, Train Acc=0.9900 | Test Loss=0.2471, Test Acc=0.9325
[BEATs] Epoch 6/10 | Train Loss=0.0675, Train Acc=0.9962 | Test Loss=0.2146, Test Acc=0.9450
[BEATs] Epoch 7/10 | Train Loss=0.0636, Train Acc=0.9944 | Test Loss=0.1989, Test Acc=0.9500
[BEATs] Epoch 8/10 | Train Loss=0.0365, Train Acc=0.9975 | Test Loss=0.1992, Test Acc=0.9475
[BEATs] Epoch 9/10 | Train Loss=0.0273, Train Acc=0.9994 | Test Loss=0.2565, Test Acc=0.9400
[BEATs] Epoch

{'model': 'BEATs',
 'checkpoint_path': '/content/drive/MyDrive/ESCProjectDeepLearning5888DataDump/checkpoints/BEATs_iter3_plus_AS2M.pt',
 'num_folds': 5,
 'mean_test_acc': np.float64(0.944),
 'std_test_acc': 0.03267261850540906,
 'mean_best_test_acc': np.float64(0.9555),
 'std_best_test_acc': 0.024520399670478456,
 'mean_test_loss': np.float64(0.21199927500030027),
 'std_test_loss': 0.12743996126933588,
 'mean_training_time_sec': np.float64(152.36683168411255)}